# Exp1034 — fill a fresh plate from prepared tubes

Use this notebook when the DNA dilution series is already prepared and mixed. Restart/close the previous experiment's kernel before opening a hardware connection here. This creates a **new session and a new run directory**; the previous run's records and counters are not reused.

The plate must be empty. Load fresh **300 µL tips in A1 through G1**, with no old tip attached. Keep the corrected deck layout. A31 supplies the blanks; A32 is unused. The six DNA sources must be A30 = 16, A29 = 8, A28 = 4, A27 = 2, A26 = 1 and A25 = 0.5 ng/µL. Buffer prefilling alone is not a prepared dilution series.

Enter **current total volumes**, not the amounts originally added. In particular, do not assume A25 contains 3000 µL: its entered volume controls its aspiration height. Each DNA tube needs at least 1500 µL, and A31 needs at least 2775 µL with the current reserve settings. If you replenish A31, enter its resulting total.

The plan fills all 96 wells using seven 300 µL tips, with 100 µL/s aspiration and 120 µL/s dispense. It performs **no preparation, mixing or refill**. Aspiration uses the calculated heights with liquid-level detection off; command completion does not verify physical delivery.

Tube aspirations are lowered by **4 mm** relative to the original calculated path. Near the end of a tube, both the starting height and the surface-following endpoint stop at **3 mm above the inner bottom**, preserving the checked 300 µL tip insertion limit and the 2 mm margin. The review table flags these limited aspirations.

Every plate dispense uses a **fixed 2.50 mm above the modeled inner well bottom**, for 50, 75, 100 and 200 µL alike. No final-volume surface offset or additional 3 mm lowering is applied. Dispense volume, flow, jet/blow-out settings and the seven-tip source grouping are unchanged. The review table shows both the inner-bottom Z and commanded tip Z.

The height reference is the modeled **inner well bottom**, including the well-bottom material thickness. The current model places it at deck Z = 184.45 mm and the dispense tip at Z = 186.95 mm. Offline checks verify commanded coordinates, not physical clearance.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import asdict
import sys
import uuid

EXPERIMENT_DIR = Path(
  "/home/hamilton-robot/Documents/hamilton-startlet-scripts/Exp1034--Filtermax-LOD"
)
PLR_CHECKOUT = Path("/home/hamilton-robot/Documents/Hamilton-Starlet/pylabrobot")
sys.path.insert(0, str(PLR_CHECKOUT))
sys.path.insert(0, str(EXPERIMENT_DIR))
import importlib
import pandas as pd
from IPython.display import display
import notebook_protocol

importlib.reload(notebook_protocol)
from notebook_protocol import ExperimentSession, Settings


## 1. Enter prepared volumes and review the offline plan

Fill in A31 and A25 below. A30–A26 are set to the reported 1500 µL per tube; edit any total that differs. Set `simulate=True` for an offline rehearsal. No instrument is connected by this cell.

In [2]:
settings = Settings(
  simulate=False,
  channel=2,
  measured_a30_concentration=16.0,
  operator="Harley",
  aspirate_flow=100.0,
  dispense_flow=120.0,
  plate_aspiration_lowering=4.0,  # mm below the original tube aspiration path, limited by clearance
  plate_dispense_height=2.50,  # fixed mm above each inner well bottom, independent of volume
)
prepared_source_volumes = {
  "A31": 5000,  # current TOTAL clean buffer, µL; at least 2775 (commonly 4000)
  "A30": 1500.0,  # prepared 16 ng/µL
  "A29": 1500.0,  # prepared 8 ng/µL
  "A28": 1500.0,  # prepared 4 ng/µL
  "A27": 1500.0,  # prepared 2 ng/µL
  "A26": 1500.0,  # prepared 1 ng/µL
  "A25": 3000,  # current TOTAL 0.5 ng/µL DNA, µL: enter 1500 or 3000 only if accurate
}
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
mode = "simulation" if settings.simulate else "physical"
output_dir = EXPERIMENT_DIR / "runs" / f"{mode}-plate-only-{run_id}"
session = ExperimentSession(
  settings, output_dir, prepared_source_volumes=prepared_source_volumes
)
print("Outputs:", output_dir)
print("Tip counts:", session.report["tip_counts"])
print("A32 is unused; its zero tracker is not a measurement of its contents.")
display(pd.DataFrame(session.ledger))
display(pd.DataFrame([
  {key: event[key] for key in (
    "source", "destination", "volume", "source_before", "aspiration_height",
    "aspiration_end_height", "aspiration_height_limited", "dispense_height", "destination_inner_bottom_z", "dispense_tip_z"
  )}
  for event in session.plan if event["action"] == "transfer"
]))
limited_aspirations = [
  event for event in session.plan
  if event.get("aspiration_height_limited", False)
]
print("Tube aspirations: 4 mm lower, stopping at the 3 mm inner-bottom/reach limit.")
print("Aspirations limited by bottom/tip clearance:", len(limited_aspirations))
print("All plate dispenses: fixed 2.50 mm above the modeled inner well bottom.")
print("Modeled plate dispense tip Z: 186.95 mm; physical clearance is not verified by simulation.")


Outputs: /home/hamilton-robot/Documents/hamilton-startlet-scripts/Exp1034--Filtermax-LOD/runs/physical-plate-only-20261006T132610Z-cc1aa2d4
Tip counts: {300: 7}
A32 is unused; its zero tracker is not a measurement of its contents.


,stage,A32,A31,A30,A29,A28,A27,A26,A25
0,operator_supplied_prepared_tubes,0,5000,1500.0,1500.0,1500.0,1500.0,1500.0,3000
1,plate_complete,0,2450,225.0,225.0,225.0,225.0,225.0,1725


,source,destination,volume,source_before,aspiration_height,aspiration_end_height,aspiration_height_limited,dispense_height,destination_inner_bottom_z,dispense_tip_z
0,A31,A9,200,5000.0,56.080611,53.730547,False,2.5,184.45,186.95
1,A31,A10,100,4800.0,53.730547,52.555515,False,2.5,184.45,186.95
2,A31,A7,75,4700.0,52.555515,51.674241,False,2.5,184.45,186.95
3,A31,A12,50,4625.0,51.674241,51.086725,False,2.5,184.45,186.95
4,A31,B5,100,4575.0,51.086725,49.911693,False,2.5,184.45,186.95
...,...,...,...,...,...,...,...,...,...,...
91,A25,C1,75,2225.0,23.473473,22.592199,False,2.5,184.45,186.95
92,A25,E5,200,2150.0,22.592199,20.242135,False,2.5,184.45,186.95
93,A25,H11,100,1950.0,20.242135,19.067103,False,2.5,184.45,186.95
94,A25,A11,75,1850.0,19.067103,18.185829,False,2.5,184.45,186.95


Tube aspirations: 4 mm lower, stopping at the 3 mm inner-bottom/reach limit.
Aspirations limited by bottom/tip clearance: 15
All plate dispenses: fixed 2.50 mm above the modeled inner well bottom.
Modeled plate dispense tip Z: 186.95 mm; physical clearance is not verified by simulation.


## 2. Start the new session

For a physical run, the prompt lists the volumes and concentrations you entered. Verify those totals, the empty plate and fresh tips, then type `APPROVE EXP1034 RUN`. This authorizes setup and this plate run under the repository's per-run approval rule.

In [3]:
await session.start()
print("Session initialized. Run the plate cell next; preparation and refill are skipped.")

2026-10-06 09:26:20,559 - pylabrobot.io.usb - INFO - Finding USB device...
2026-10-06 09:26:20,570 - pylabrobot.io.usb - INFO - Found USB device.
2026-10-06 09:26:20,574 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0
2026-10-06 09:26:23,842 - pylabrobot - INFO - Running backend initialization procedure.


Session initialized. Run the plate cell next; preparation and refill are skipped.


## 3. Fill the plate once

Do not run this cell on a partially filled plate. Inspect liquid uptake and delivery, especially the first aspirations from A25. After completion, check the physical plate before interpreting the run as successful. Tracked volumes describe commanded transfers.

In [4]:
await session.execute_phase("plate")
display(pd.DataFrame([session.volumes()], index=["final tracked source volumes"]))
print("Well dispense commands completed:", len(session.metadata["per_well_dispensing_timestamps"]))
print("Run records:", output_dir)
await session.stop()

plate: complete (PHYSICAL RUN).


,A32,A31,A30,A29,A28,A27,A26,A25
final tracked source volumes,0,2450.0,225.0,225.0,225.0,225.0,225.0,1725.0


2026-10-06 09:48:30,026 - pylabrobot.io.usb - WARNING - Closing connection to USB device.


Well dispense commands completed: 96
Run records: /home/hamilton-robot/Documents/hamilton-startlet-scripts/Exp1034--Filtermax-LOD/runs/physical-plate-only-20261006T132610Z-cc1aa2d4
